# Sistema RAG — Pre-entrega 3: AI Engineering

**Flujo End-to-End:**
1. **Ingesta**: Carga documentos `.txt/.md` → chunking con `RecursiveCharacterTextSplitter` → embeddings → ChromaDB
2. **Retrieval**: Convierte la pregunta en embedding → busca top-k fragmentos similares
3. **Generación Grounded**: Cadena LCEL con prompt de veracidad → LLM → `PydanticOutputParser`

**Proveedor LLM:** Ollama (local, por defecto) | OpenAI (opcional, configurar en `.env`)

## 0. Configuración del entorno

In [ ]:
# Asegurarse de estar en el directorio raíz del proyecto
import os
import sys

# Agrega el directorio raíz al path para importar los módulos
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print(f"Directorio de trabajo: {os.getcwd()}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

provider = os.getenv("LLM_PROVIDER", "ollama")
print(f"Proveedor: {provider}")
print(f"Modelo LLM: {os.getenv('OLLAMA_MODEL', 'llama3.2') if provider == 'ollama' else os.getenv('OPENAI_MODEL', 'gpt-4o-mini')}")
print(f"Modelo embeddings: {os.getenv('OLLAMA_EMBEDDING_MODEL', 'nomic-embed-text') if provider == 'ollama' else 'text-embedding-3-small'}")

## 1. Módulo de Ingesta — Chunking y ChromaDB

Lee los 4 archivos `.txt` de `/data`, los fragmenta en chunks de **500 tokens** con **50 tokens de overlap** usando `RecursiveCharacterTextSplitter`, genera embeddings y los persiste en ChromaDB.

> Si el vectorstore ya existe, se carga sin re-indexar (optimización de costo).

In [ ]:
from ingest import ingest_documents, CHUNK_SIZE, CHUNK_OVERLAP

print(f"Parámetros de chunking: chunk_size={CHUNK_SIZE} tokens, overlap={CHUNK_OVERLAP} tokens")
vectorstore = ingest_documents()
print("\nVectorstore listo.")

In [ ]:
# Verificar cuántos documentos hay en la colección
import chromadb

client = chromadb.PersistentClient(path="./vectorstore")
collection = client.get_collection("rag_documents")
print(f"Total de chunks en ChromaDB: {collection.count()}")

## 2. Capa de Recuperación — Retriever

Configura el retriever con `top_k=4` para recuperar los 4 fragmentos más relevantes. El mismo modelo de embeddings usado en la ingesta es el que convierte la query.

In [ ]:
from rag_chain import TOP_K
from pathlib import Path

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
print(f"Retriever configurado con top_k={TOP_K}")

# Test rápido del retriever
test_query = "¿Qué son los embeddings?"
docs = retriever.invoke(test_query)
print(f"\nQuery de prueba: '{test_query}'")
print(f"Fragmentos recuperados: {len(docs)}")
for i, doc in enumerate(docs, 1):
    src = Path(doc.metadata.get('source', '')).name
    print(f"  [{i}] {src} — {doc.page_content[:100]}...")

## 3. Cadena RAG con LCEL

La cadena une:
- **Retriever** → fragmentos relevantes
- **`format_docs`** → texto formateado con fuentes
- **Prompt** con instrucción de veracidad ("di que no lo sabes si no está en el contexto")
- **LLM** (Ollama/OpenAI) → respuesta en JSON
- **`PydanticOutputParser`** → `RAGResponse` con `answer`, `sources` y `confidence`

In [ ]:
from rag_chain import build_rag_chain, get_rag_response, SYSTEM_PROMPT

chain = build_rag_chain(retriever)
print("Cadena LCEL construida.")
print("\n--- Prompt del sistema (extracto) ---")
print(SYSTEM_PROMPT[:400] + "...")

## 4. Prueba 1 — Pregunta cuya respuesta SÍ está en los documentos

In [ ]:
import asyncio
import textwrap

query_1 = "¿Cómo funciona el mecanismo de atención (self-attention) en los Transformers?"

print(f"Pregunta: {query_1}\n")
response_1 = await get_rag_response(query_1, retriever, chain)

print("=" * 60)
print(f"CONFIANZA: {response_1.confidence}")
print("\nRESPUESTA:")
print(textwrap.fill(response_1.answer, width=70))

if response_1.sources:
    print(f"\nFRAGMENTOS USADOS ({len(response_1.sources)}):")
    for i, src in enumerate(response_1.sources, 1):
        print(f"  [{i}] {src[:150]}...")
print("=" * 60)

## 5. Prueba 2 — Pregunta trampa (respuesta NO está en los documentos)

El modelo debe responder que **no tiene esa información** en los documentos, sin alucinar.

In [ ]:
query_2 = "¿Cuál es la receta tradicional de la paella valenciana y cuántos ingredientes lleva?"

print(f"Pregunta: {query_2}\n")
response_2 = await get_rag_response(query_2, retriever, chain)

print("=" * 60)
print(f"CONFIANZA: {response_2.confidence}")
print("\nRESPUESTA:")
print(textwrap.fill(response_2.answer, width=70))
print("=" * 60)

# Verificación
no_info_signals = [
    response_2.confidence == "sin_informacion",
    "no tengo" in response_2.answer.lower(),
    "no acceso" in response_2.answer.lower(),
    "no está" in response_2.answer.lower(),
    "no encuentro" in response_2.answer.lower(),
]
passed = any(no_info_signals)
print(f"\n{'✅ El modelo reconoció que no tiene esa información.' if passed else '❌ El modelo puede estar alucinando — revisar el prompt.'}")

## 6. Prueba adicional — Pregunta sobre ML dentro del contexto

In [ ]:
query_3 = "¿Qué es el overfitting y cómo se puede prevenir con regularización?"

print(f"Pregunta: {query_3}\n")
response_3 = await get_rag_response(query_3, retriever, chain)

print("=" * 60)
print(f"CONFIANZA: {response_3.confidence}")
print("\nRESPUESTA:")
print(textwrap.fill(response_3.answer, width=70))
print("=" * 60)

## Resumen

| Componente | Implementación |
|---|---|
| Chunking | `RecursiveCharacterTextSplitter` (500 tokens, 50 overlap) |
| Vector DB | ChromaDB persistente en `./vectorstore` |
| Embeddings | `nomic-embed-text` (Ollama) / `text-embedding-3-small` (OpenAI) |
| LLM | `llama3.2` (Ollama) / `gpt-4o-mini` (OpenAI) |
| Framework | LangChain LCEL |
| Output | `PydanticOutputParser` → `RAGResponse` |
| top_k | 4 fragmentos (evita "Lost in the Middle") |